In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from concurrent.futures import ThreadPoolExecutor
import os
from sklearn.preprocessing import StandardScaler

os.environ["OMP_NUM_THREADS"] = "1"

# -------------------------- 核心函数 --------------------------
def compute_distances(X, s, n_jobs=8):
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    chunks = np.array_split(indices, n_jobs)
    ranges = [(chunk[0], chunk[-1]+1) for chunk in chunks]
    
    distance_matrix = np.zeros((n_samples, n_samples))
    
    with ThreadPoolExecutor(max_workers=n_jobs) as executor:
        futures = []
        for i in range(len(ranges)):
            for j in range(i, len(ranges)):
                start_i, end_i = ranges[i]
                start_j, end_j = ranges[j]
                
                future = executor.submit(
                    lambda a, b, X: cdist(X[a[0]:a[1]], X[b[0]:b[1]], 'sqeuclidean'),
                    (start_i, end_i), (start_j, end_j), X
                )
                futures.append( (future, i, j) )
        
        for future, i, j in futures:
            start_i, end_i = ranges[i]
            start_j, end_j = ranges[j]
            block = future.result()
            
            # 填充主块
            distance_matrix[start_i:end_i, start_j:end_j] = block
            # 填充对称块
            if i != j:
             distance_matrix[start_j:end_j, start_i:end_i] = block.T
    #kernel_matrix = np.exp(-distance_matrix / (2 * s**2))  # 应用高斯核公式
    
    return distance_matrix # 返回计算好的高斯核矩阵


# -------------------------- 完整工作流 --------------------------
if __name__ == "__main__":
    # 1. 数据加载与标准化
    expression_matrix = pd.read_csv("/Users/stayy/expression_matrix_common.csv")
    X = expression_matrix.values.T
    
    # 2. 自动带宽选择
    distance_matrix=compute_distances(X,1,n_jobs=8)
    gamma = 1 / (2 * np.median(distance_matrix))
    # 3. 计算高斯核矩阵
    Q = np.exp(-gamma*distance_matrix)
    
    # 4. 验证非零元素
    print(f"非零元素比例: {np.mean(Q > 1e-6)*100:.2f}%")
    print("示例输出（左上5x5）:\n", np.round(Q[:5, :5], 4))
    D = np.diag(np.sum(Q, axis=1))
    # 计算拉普拉斯矩阵 L
    L = D - Q

# 输出计算结果
print("Q (高斯核矩阵):\n", Q[:5, :5])  # 仅显示前5行5列
print("D (度矩阵):\n", D[:5, :5])  # 仅显示前5行5列
print("L (拉普拉斯矩阵):\n", L[:5, :5])  # 仅显示前5行5列

In [ ]:
import numpy as np

def scANMF(X, A, rank, Marker, alpha, beta, lambda_, L, D, Q, iteration, sigma):
    # 设置矩阵的维度
    num_rows_W = X.shape[0]
    num_cols_W = rank
    num_rows_H = X.shape[1]
    num_cols_H = rank
    
    # 生成随机初始化的矩阵 W 和 H
    W = np.random.rand(num_rows_W, num_cols_W)
    H = np.random.rand(num_rows_H, num_cols_H)
    
    # 初始化变量
    Ti = 0
    cha = 1
    tt = []  # 存储每次迭代的目标函数值
    objective = np.linalg.norm(X - np.dot(W, H.T), 'fro')**2 + alpha * np.sum(W * Marker) + beta * np.sum(np.diag(np.dot(H.T, np.dot(L, H)))) + lambda_ * np.sum(H * A)    
    try:
        while np.abs(cha) > sigma and Ti < iteration:
            print(np.linalg.norm(X - np.dot(W, H.T), 'fro')**2)
            print (alpha * np.sum(W * Marker))
            print(beta * np.sum(np.diag(np.dot(H.T, np.dot(L, H))))) 
            print(lambda_ * np.sum(H * A) )
            XH = np.dot(X, H) 
            WHtH = np.dot(W, np.dot(H.T, H))
            W = W * (2 * XH) / ((2 * WHtH + alpha * Marker)+1e-10) # 添加一个小的常数避免除零错误
            H = H * (2 * (np.dot(X.T, W) + 2 * beta * np.dot(Q, H))) /  (2 * (np.dot(H, np.dot(W.T, W)) + 2 * beta * np.dot(D, H)) + lambda_ * A + 1e-10)
            new_objective = np.linalg.norm(X - np.dot(W, H.T), 'fro')**2 + alpha * np.sum(W * Marker) + beta * np.sum(np.diag(np.dot(H.T, np.dot(L, H)))) + lambda_ * np.sum(H * A)    
            # 将当前的目标函数值添加到 tt 列表中
            tt.append(new_objective)
            cha = new_objective - objective
            print(cha)
            objective = new_objective
            Ti += 1
    except Exception as e:
        print("❌ 发生错误，迭代中断！")
        print(f"⏳ 迭代终止于 Ti={Ti}")
        print(f"⚠️ 错误信息：{e}")
        print("🔹 当前 H 矩阵（部分）：\n", H)
    return W, H, objective, tt, Ti, cha

In [ ]:
#交叉验证
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import numpy as np
expression_matrix=pd.read_csv("/Users/stayy/expression_matrix_common.csv")
labelMatrix = pd.read_csv("/Users/stayy/labelMatrix.csv").values
column_names = pd.read_csv("/Users/stayy/labelMatrix.csv").columns.values
Markers = pd.read_csv("/Users/stayy/Markers.csv").values
Marker = np.array(Markers, dtype=float)  # 确保转换为 NumPy 数组
labelMatrix= np.array(labelMatrix, dtype=float)   # 确保 labelMatrix 也是数组
# 假设你有部分标签数据，你可以使用KFold进行交叉验证
kf = KFold(n_splits=5, shuffle=True, random_state=42)
# 网格搜索参数范围
alpha_values = [1]  
beta_values = [0.1] 
lambda_values = [2,4,6,8]  
iterations = 500
tolerance = 0.1
expression_matrix_kfold = expression_matrix.iloc[0:3000, 4218:14819]  # 选择第 1 到第 2 行和第 2 到第 3 列
expression_matrix_kfold
labelMatrix_kfold = labelMatrix[4218:14819, 0:3000]  
labelMatrix_kfold
label_Pancreas_test = pd.read_csv("/Users/stayy/Pictures/lable_yidao.csv", header=None)

# 选择第一列
label_Pancreas_test = label_Pancreas_test.iloc[:, 0]

# 先用 replace() 处理替换内容（在 Pandas Series 阶段进行）
label_Pancreas_test = label_Pancreas_test.replace({'PSC': 'Pancreatic stellate', 'MHC class II': 'unknown'})

# 再把所有元素的首字母改成大写（用 .str.capitalize() ）
label_Pancreas_test = label_Pancreas_test.str.capitalize()

# 在每个单词后面加 "cells"
label_Pancreas_test = label_Pancreas_test + " cells"

# 继续替换特殊情况
label_Pancreas_test = label_Pancreas_test.replace({'Pp cells': 'PP cells', 'Unknown cells': 'unknown'})

# 初始化记录网格搜索结果的列表
grid_search_results = []
# 进行网格搜索
for alpha in alpha_values:
    for beta in beta_values:
        for lambda_ in lambda_values:
            fold_accuracies = []  # 记录每一折的准确度
            
            # 对数据进行交叉验证
            for train_index, test_index in kf.split(expression_matrix_kfold):
                # 划分训练集和测试集
                labelMatrix_kfold_test = labelMatrix_kfold.copy()  # 复制以免修改原数据
                labelMatrix_kfold_test[test_index, :] = 0  # 将 test_index 对应的行全变成 0
                
                # 调用 scANMF 函数进行训练
                result= scANMF(expression_matrix_kfold, labelMatrix_kfold_test,labelMatrix_kfold_test.shape[1], Marker, alpha, beta, lambda_, L_kfold, D_kfold, Q_kfold, iterations, tolerance)
                
                # 预测并计算准确度
                celltype_matrix= result[1]

                dfkfold= pd.DataFrame(celltype_matrix[test_index], columns=column_names)
                max_col_names = dfkfold.idxmax(axis=1) # 预测的细胞类型
                label_true = label_Pancreas_train.iloc[test_index].values.flatten()

                # 比较预测与真实标签的准确性
                score = (max_col_names == label_true).mean()
                print(score)
                fold_accuracies.append(score)# 将准确度添加到 fold_accuracies 中
                
            # 计算每个参数组合的平均准确度
            mean_accuracy = np.mean(fold_accuracies)
            grid_search_results.append({
                'alpha': alpha,
                'beta': beta,
                'lambda_': lambda_,
                'mean_accuracy': mean_accuracy
            })

# 将结果转换为 DataFrame
grid_search_df = pd.DataFrame(grid_search_results)
print(grid_search_df)

In [ ]:


result111= scANMF(expression_matrix, labelMatrix, labelMatrix.shape[1], Marker, 1000, 0.001, 10, L, D, Q, 500, 0.01)
cell_type_matrix111=result111[1]
gene_matrix111=result111[0]

df111= pd.DataFrame(cell_type_matrix111, columns=column_names)

# 获取每行最大值所在的列名
max_col_names111 = df111.iloc[:4218].apply(lambda row: row.idxmax(), axis=1)
equal_count111 = (label_Pancreas_test == max_col_names111).sum()

print(f"完全相等的个数: {equal_count111}")
